In [3]:
# Conditional Sequential VAE for 60-month yield-curve scenario generation
# Paste this entire block into VS Code and run.
#
# Expected input file:
# raw_data.csv
#
# Expected columns:
# DATE, Y_DGS1MO, Y_DGS3MO, Y_DGS6MO, Y_DGS1, Y_DGS2, Y_DGS3,
# Y_DGS5, Y_DGS7, Y_DGS10, Y_DGS20, Y_DGS30

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# =========================
# CONFIG
# =========================

FILE_PATH = "raw_data.csv"
OUTPUT_FILE = "generated_yield_curve_scenarios.csv"

YIELD_COLS = [
    "Y_DGS1MO", "Y_DGS3MO", "Y_DGS6MO",
    "Y_DGS1", "Y_DGS2", "Y_DGS3", "Y_DGS5",
    "Y_DGS7", "Y_DGS10", "Y_DGS20", "Y_DGS30"
]

HORIZON = 60
N_SCENARIOS = 200
LATENT_DIM = 16
HIDDEN_DIM = 256
BATCH_SIZE = 32
EPOCHS = 500
LEARNING_RATE = 1e-3

BETA_KL = 0.01
SMOOTH_LAMBDA = 0.10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)

# =========================
# LOAD DATA
# =========================

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"Could not find input file: {FILE_PATH}")

try:
    df = pd.read_csv(FILE_PATH)
except Exception:
    df = pd.read_excel(FILE_PATH)

df["DATE"] = pd.to_datetime(df["DATE"])
df = df[["DATE"] + YIELD_COLS].copy()

for col in YIELD_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna().sort_values("DATE")

monthly = (
    df.set_index("DATE")[YIELD_COLS]
    .resample("M")
    .last()
    .dropna()
)

if len(monthly) <= HORIZON:
    raise ValueError(
        f"Not enough monthly data. Need more than {HORIZON} months, got {len(monthly)}."
    )

curves = monthly.values.astype("float32")
dates = monthly.index

# =========================
# CREATE TRAINING SAMPLES
# =========================

X = []
y = []

for i in range(len(curves) - HORIZON):
    current_curve = curves[i]

    future_curves = curves[i + 1 : i + HORIZON + 1]
    previous_curves = curves[i : i + HORIZON]

    monthly_changes = future_curves - previous_curves

    X.append(current_curve)
    y.append(monthly_changes)

X = np.array(X, dtype="float32")
y = np.array(y, dtype="float32")

print("Current curve input shape:", X.shape)
print("Future change target shape:", y.shape)

# =========================
# NORMALIZATION
# =========================

x_mean = X.mean(axis=0)
x_std = X.std(axis=0) + 1e-6

y_mean = y.reshape(-1, len(YIELD_COLS)).mean(axis=0)
y_std = y.reshape(-1, len(YIELD_COLS)).std(axis=0) + 1e-6

X_scaled = (X - x_mean) / x_std
y_scaled = (y - y_mean) / y_std

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# =========================
# MODEL
# =========================

class ConditionalYieldCurveVAE(nn.Module):
    def __init__(self, n_tenors, horizon=60, latent_dim=16, hidden_dim=256):
        super().__init__()

        self.n_tenors = n_tenors
        self.horizon = horizon
        self.output_dim = horizon * n_tenors

        encoder_input_dim = n_tenors + self.output_dim

        self.encoder = nn.Sequential(
            nn.Linear(encoder_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

        decoder_input_dim = n_tenors + latent_dim

        self.decoder = nn.Sequential(
            nn.Linear(decoder_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, self.output_dim),
        )

    def encode(self, y0, future_changes):
        batch_size = y0.shape[0]
        flat_future = future_changes.reshape(batch_size, -1)
        x = torch.cat([y0, flat_future], dim=1)
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, y0, z):
        x = torch.cat([y0, z], dim=1)
        out = self.decoder(x)
        return out.reshape(-1, self.horizon, self.n_tenors)

    def forward(self, y0, future_changes):
        mu, logvar = self.encode(y0, future_changes)
        z = self.reparameterize(mu, logvar)
        pred_changes = self.decode(y0, z)
        return pred_changes, mu, logvar

# =========================
# LOSS
# =========================

def vae_loss(pred, target, mu, logvar):
    reconstruction_loss = F.mse_loss(pred, target)

    kl_loss = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    # Smoothness across maturity tenors
    tenor_diff = pred[:, :, 1:] - pred[:, :, :-1]
    smoothness_loss = torch.mean(tenor_diff ** 2)

    total_loss = (
        reconstruction_loss
        + BETA_KL * kl_loss
        + SMOOTH_LAMBDA * smoothness_loss
    )

    return total_loss, reconstruction_loss, kl_loss, smoothness_loss

# =========================
# TRAIN
# =========================

model = ConditionalYieldCurveVAE(
    n_tenors=len(YIELD_COLS),
    horizon=HORIZON,
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_epoch_loss = 0.0

    for y0_batch, changes_batch in train_loader:
        y0_batch = y0_batch.to(DEVICE)
        changes_batch = changes_batch.to(DEVICE)

        pred, mu, logvar = model(y0_batch, changes_batch)

        loss, recon, kl, smooth = vae_loss(pred, changes_batch, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    if epoch % 50 == 0 or epoch == 1:
        avg_loss = total_epoch_loss / len(train_loader)
        print(f"Epoch {epoch:04d} | Loss: {avg_loss:.6f}")

# =========================
# GENERATE SCENARIOS
# =========================

def generate_scenarios(model, current_curve, n_scenarios=200):
    model.eval()

    current_curve = np.array(current_curve, dtype="float32")
    current_scaled = (current_curve - x_mean) / x_std

    y0 = torch.tensor(current_scaled, dtype=torch.float32)
    y0 = y0.unsqueeze(0).repeat(n_scenarios, 1).to(DEVICE)

    z = torch.randn(n_scenarios, LATENT_DIM).to(DEVICE)

    with torch.no_grad():
        pred_scaled_changes = model.decode(y0, z).cpu().numpy()

    pred_changes = pred_scaled_changes * y_std + y_mean

    projected_curves = (
        current_curve.reshape(1, 1, len(YIELD_COLS))
        + np.cumsum(pred_changes, axis=1)
    )

    return projected_curves

current_curve = monthly.iloc[-1].values.astype("float32")
last_date = monthly.index[-1]

scenarios = generate_scenarios(
    model=model,
    current_curve=current_curve,
    n_scenarios=N_SCENARIOS
)

print("Generated scenario shape:", scenarios.shape)

# =========================
# EXPORT CSV
# =========================

projection_dates = pd.date_range(
    last_date + pd.offsets.MonthEnd(1),
    periods=HORIZON,
    freq="M"
)

rows = []

for scenario_id in range(N_SCENARIOS):
    for month_idx in range(HORIZON):
        row = {
            "scenario_id": scenario_id + 1,
            "projection_month": month_idx + 1,
            "projection_date": projection_dates[month_idx].date()
        }

        for tenor_idx, col in enumerate(YIELD_COLS):
            row[col] = scenarios[scenario_id, month_idx, tenor_idx]

        rows.append(row)

scenario_df = pd.DataFrame(rows)
scenario_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved generated scenarios to: {OUTPUT_FILE}")

# =========================
# SIMPLE VALIDATION SUMMARY
# =========================

historical_changes = y.reshape(-1, len(YIELD_COLS))
generated_changes = np.diff(
    np.concatenate(
        [
            current_curve.reshape(1, 1, len(YIELD_COLS)).repeat(N_SCENARIOS, axis=0),
            scenarios
        ],
        axis=1
    ),
    axis=1
).reshape(-1, len(YIELD_COLS))

print("\nValidation summary")
print("------------------")
print("Historical monthly change std by tenor:")
print(pd.Series(historical_changes.std(axis=0), index=YIELD_COLS))

print("\nGenerated monthly change std by tenor:")
print(pd.Series(generated_changes.std(axis=0), index=YIELD_COLS))

print("\nLatest input curve:")
print(pd.Series(current_curve, index=YIELD_COLS))

print("\nFirst generated scenario, first 5 months:")
print(scenario_df[scenario_df["scenario_id"] == 1].head())

C:\Users\calvi\AppData\Local\Temp\ipykernel_12248\2807818939.py:70: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample("M")


Current curve input shape: (230, 11)
Future change target shape: (230, 60, 11)
Epoch 0001 | Loss: 1.011840
Epoch 0050 | Loss: 0.357957
Epoch 0100 | Loss: 0.165117
Epoch 0150 | Loss: 0.107533
Epoch 0200 | Loss: 0.088018
Epoch 0250 | Loss: 0.076845
Epoch 0300 | Loss: 0.071808
Epoch 0350 | Loss: 0.061643
Epoch 0400 | Loss: 0.061679
Epoch 0450 | Loss: 0.058931
Epoch 0500 | Loss: 0.059056
Generated scenario shape: (200, 60, 11)


C:\Users\calvi\AppData\Local\Temp\ipykernel_12248\2807818939.py:284: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  projection_dates = pd.date_range(


Saved generated scenarios to: generated_yield_curve_scenarios.csv

Validation summary
------------------
Historical monthly change std by tenor:
Y_DGS1MO    0.262424
Y_DGS3MO    0.209918
Y_DGS6MO    0.198918
Y_DGS1      0.197085
Y_DGS2      0.221872
Y_DGS3      0.237107
Y_DGS5      0.252047
Y_DGS7      0.254530
Y_DGS10     0.248682
Y_DGS20     0.239145
Y_DGS30     0.229871
dtype: float32

Generated monthly change std by tenor:
Y_DGS1MO    0.183805
Y_DGS3MO    0.143956
Y_DGS6MO    0.137229
Y_DGS1      0.132643
Y_DGS2      0.146424
Y_DGS3      0.156160
Y_DGS5      0.165805
Y_DGS7      0.167536
Y_DGS10     0.164487
Y_DGS20     0.160554
Y_DGS30     0.157399
dtype: float32

Latest input curve:
Y_DGS1MO    3.72
Y_DGS3MO    3.69
Y_DGS6MO    3.61
Y_DGS1      3.51
Y_DGS2      3.48
Y_DGS3      3.50
Y_DGS5      3.65
Y_DGS7      3.85
Y_DGS10     4.08
Y_DGS20     4.66
Y_DGS30     4.72
dtype: float32

First generated scenario, first 5 months:
   scenario_id  projection_month projection_date  Y_DGS1M

In [13]:
import pandas as pd

# 1. Load the CSV file into a DataFrame
file_name = 'generated_yield_curve_scenarios.csv'
df = pd.read_csv(file_name)

# 2. Drop the 'Date' column
# axis=1 specifies that we are dropping a column (axis=0 would be a row)
# We can create a new DataFrame or use inplace=True to modify the original
#df_cleaned = df.drop('projection_date', axis=1)

df = df.rename(columns={'Month': 'projection_month', 'Scenario_ID': 'scenario_id'})
# Alternatively, you can use the 'columns' parameter for better readability:
# df_cleaned = df.drop(columns=['Date'])

# 3. Save the modified DataFrame to a new CSV file
df_cleaned.to_csv('generated_yield_curve_scenarios.csv', index=False)

# Display the first few rows to verify the column is gone
print(df_cleaned.head())

   scenario_id  projection_month  Y_DGS1MO  Y_DGS3MO  Y_DGS6MO    Y_DGS1  \
0            1                 1  3.854937  3.583663  3.617837  3.594678   
1            1                 2  3.293157  3.297604  3.314469  3.304227   
2            1                 3  3.239184  3.304334  3.331847  3.326696   
3            1                 4  3.368306  3.202295  3.159950  3.145232   
4            1                 5  2.785101  2.683853  2.795271  2.833351   

     Y_DGS2    Y_DGS3    Y_DGS5    Y_DGS7   Y_DGS10   Y_DGS20   Y_DGS30  
0  3.683326  3.805922  3.959737  4.144860  4.250415  4.771738  4.788338  
1  3.443146  3.565541  3.725372  3.944783  4.103871  4.645948  4.672123  
2  3.441794  3.584280  3.733539  3.976677  4.188954  4.776707  4.851577  
3  3.239998  3.419224  3.663030  3.983410  4.173075  4.777544  4.819615  
4  2.995494  3.270417  3.706492  4.069744  4.348969  4.940821  4.971292  
